# UdaPlay — Part 1: RAG Pipeline

Build and verify the knowledge base for the UdaPlay gaming research agent.

## What this notebook does

1. **Load** 25 game records from `games.json`
2. **Embed** each record with OpenAI `text-embedding-ada-002` (via ChromaDB)
3. **Store** in a ChromaDB in-memory collection
4. **Query** to verify semantic search works end-to-end

The resulting `VectorStore` object is reused in `Udaplay_02_solution_project.ipynb`.

## Framework

This project follows the patterns from the *Building Agents* course:
- `lib.vector_db.VectorStoreManager` — manages ChromaDB collections
- `lib.vector_db.CorpusLoaderService` — loads datasets into vector stores
- `lib.loaders.JSONGameLoader` — converts JSON game records to `Document` objects
- `lib.state_machine.StateMachine` — powers the RAG retrieve→augment→generate loop
- `lib.rag.RAG` — the complete RAG pipeline as a reusable component

In [ ]:
# Uncomment if dependencies are not yet installed
# !pip install chromadb>=1.0.4 openai>=1.73.0 pydantic>=2.11.3 python-dotenv>=1.1.0 tavily-python>=0.5.4 pdfplumber

In [ ]:
# Only needed for Udacity workspace
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY must be set in config.env"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY must be set in config.env"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

print("✓ Environment loaded")
print(f"  Base URL: {OPENAI_BASE_URL}")

In [ ]:
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG
from lib.llm import LLM
from lib.state_machine import Run

print("✓ lib imports successful")

## Step 1 — Initialize the Vector Store Manager

In [ ]:
# VectorStoreManager handles ChromaDB client setup and OpenAI embedding function.
# Pass api_base so the embedding calls route through the Vocareum proxy.

db = VectorStoreManager(OPENAI_API_KEY, api_base=OPENAI_BASE_URL)
print(db)

In [ ]:
loader_service = CorpusLoaderService(db)
print("✓ CorpusLoaderService ready")

## Step 2 — Load and embed the games dataset

`CorpusLoaderService.load_json()` uses `JSONGameLoader` under the hood:
1. Reads every game record from `games.json`
2. Converts each record into a natural-language `Document`
3. Batches all documents into a ChromaDB collection (OpenAI embeddings are generated automatically)

In [ ]:
games_store = loader_service.load_json(
    store_name="games",
    json_path="games.json",
)

## Step 3 — Verify semantic search

Query the vector store directly and inspect similarity scores.

In [ ]:
import json

def print_search_results(query: str, n_results: int = 3) -> None:
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    raw = games_store.query(query_texts=[query], n_results=n_results)
    docs  = raw["documents"][0] if raw["documents"] else []
    dists = raw["distances"][0] if raw["distances"] else []
    metas = raw["metadatas"][0] if raw["metadatas"] else []

    for i, (doc, dist, meta) in enumerate(zip(docs, dists, metas), 1):
        title = meta.get("title", "Unknown")
        dev   = meta.get("developer", "Unknown")
        year  = meta.get("release_date", "?")[:4]
        print(f"  {i}. [similarity {1-dist:.4f}] {title} — {dev} ({year})")


queries = [
    "Who developed FIFA 21?",
    "What platform was Pokemon Red originally launched on?",
    "When was God of War Ragnarok released?",
    "Open-world action RPG games",
    "Games developed by Rockstar Games",
    "First-person shooter with campaign mode",
    "Nintendo exclusive life simulation games",
]

for q in queries:
    print_search_results(q)

## Step 4 — RAG pipeline demo

Wrap the vector store in a `RAG` pipeline (retrieve → augment → generate)
to produce natural-language answers backed by the retrieved context.

In [ ]:
rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

games_rag = RAG(
    llm=rag_llm,
    vector_store=games_store,
)

print("✓ RAG pipeline ready")

In [ ]:
# RAG query 1 — developer lookup
result: Run = games_rag.invoke("Who developed FIFA 21?")
print(result.get_final_state()["answer"])

In [ ]:
# RAG query 2 — platform and release date
result: Run = games_rag.invoke(
    "What platform was Pokémon Red originally launched on and in what year?"
)
print(result.get_final_state()["answer"])

In [ ]:
# RAG query 3 — genre-based discovery
result: Run = games_rag.invoke(
    "List the open-world RPG games in the dataset with their release dates."
)
print(result.get_final_state()["answer"])

## Step 5 — Dataset statistics

In [ ]:
raw_all = games_store.get(limit=100)
all_metas = raw_all["metadatas"] or []
all_games = [json.loads(m["json_data"]) for m in all_metas]

print(f"Total games indexed: {len(all_games)}")

genre_counts: dict = {}
year_counts: dict = {}
for g in all_games:
    genre_counts[g.get("genre", "Unknown")] = genre_counts.get(g.get("genre", "Unknown"), 0) + 1
    year = g.get("release_date", "?")[:4]
    year_counts[year] = year_counts.get(year, 0) + 1

print("\nGenre breakdown:")
for genre, count in sorted(genre_counts.items(), key=lambda x: -x[1]):
    print(f"  {genre:<40} {'█' * count} ({count})")

print("\nRelease year distribution:")
for year in sorted(year_counts):
    print(f"  {year}  {'█' * year_counts[year]} ({year_counts[year]})")

print("\n✓ RAG pipeline verified — run Udaplay_02_solution_project.ipynb for the agent.")